# Data Cleaning, Table Integration and Executive EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/02_data_cleaning_and_eda.ipynb)

This notebook documents the complete path from three raw Kaggle tables to clean business metrics and five-year visual analysis.

> **Public portfolio snapshot:** All tables, metrics and charts below were generated from the official Kaggle M5 data and saved in this notebook. You can review the complete work without credentials. To reproduce the analysis, run the notebook with your own Kaggle API token; no private token is stored in this repository.

## 1. Environment and official data

In [1]:
import sys, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=1.0,<2', 'pandas>=2.2,<3', 'numpy>=2,<3',
        'plotly>=5.24,<7', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5'
    ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', 30)


In [2]:
import kagglehub

# Download latest version. KaggleHub automatically checks the Colab secret
# named KAGGLE_API_TOKEN. If it is missing, the login widget opens once.
try:
    path = kagglehub.competition_download('m5-forecasting-accuracy')
except Exception as error:
    if error.__class__.__name__ != 'UnauthenticatedError':
        raise
    print('Kaggle authentication is required. Paste your Kaggle API token in the login form below.')
    kagglehub.login()
    path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

DATA_DIR = Path(path)
if not (DATA_DIR / 'calendar.csv').exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob('calendar.csv'))

required = {'calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv'}
available = {p.name for p in DATA_DIR.glob('*.csv')}
assert required.issubset(available), f"Missing files: {sorted(required - available)}"


Official Kaggle M5 competition files loaded successfully for this public snapshot.


## 2. Load raw tables and validate business keys

In [3]:
sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
calendar = pd.read_csv(DATA_DIR / 'calendar.csv', parse_dates=['date'])
prices = pd.read_csv(DATA_DIR / 'sell_prices.csv')
day_cols = [c for c in sales if c.startswith('d_')]

checks = pd.Series({
    'duplicate_sales_ids': sales.id.duplicated().sum(),
    'duplicate_calendar_days': calendar.d.duplicated().sum(),
    'duplicate_price_keys': prices.duplicated(['store_id', 'item_id', 'wm_yr_wk']).sum(),
    'missing_prices': prices.sell_price.isna().sum(),
    'negative_prices': (prices.sell_price < 0).sum(),
})
display(checks.to_frame('count'))
assert checks.sum() == 0, 'Review failed quality checks before continuing.'


,count
duplicate_sales_ids,0
duplicate_calendar_days,0
duplicate_price_keys,0
missing_prices,0
negative_prices,0


## 3. Explain the relational model
`d` connects demand to calendar. Calendar contributes `wm_yr_wk`, which combines with store and item to identify the correct weekly price.

In [4]:
id_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
preview = (
    sales.loc[:99, id_cols + day_cols]
    .melt(id_vars=id_cols, var_name='d', value_name='units')
    .merge(calendar, on='d', how='left', validate='many_to_one')
    .merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left', validate='many_to_one')
)
preview['estimated_revenue'] = preview.units * preview.sell_price
display(preview[['date', 'd', 'store_id', 'item_id', 'cat_id', 'units', 'sell_price', 'estimated_revenue']].head(10))
print('Preview rows:', f'{len(preview):,}', '| Missing dates:', preview.date.isna().sum())


,date,d,store_id,item_id,cat_id,units,sell_price,estimated_revenue
0,2011-01-29,d_1,CA_1,HOBBIES_1_001,HOBBIES,0,NaN,NaN
1,2011-01-29,d_1,CA_1,HOBBIES_1_002,HOBBIES,0,NaN,NaN
2,2011-01-29,d_1,CA_1,HOBBIES_1_003,HOBBIES,0,NaN,NaN
3,2011-01-29,d_1,CA_1,HOBBIES_1_004,HOBBIES,0,NaN,NaN
4,2011-01-29,d_1,CA_1,HOBBIES_1_005,HOBBIES,0,NaN,NaN
5,2011-01-29,d_1,CA_1,HOBBIES_1_006,HOBBIES,0,NaN,NaN
6,2011-01-29,d_1,CA_1,HOBBIES_1_007,HOBBIES,0,NaN,NaN
7,2011-01-29,d_1,CA_1,HOBBIES_1_008,HOBBIES,12,0.46,5.52
8,2011-01-29,d_1,CA_1,HOBBIES_1_009,HOBBIES,2,1.56,3.12
9,2011-01-29,d_1,CA_1,HOBBIES_1_010,HOBBIES,0,3.17,0.00


Preview rows: 194,100 | Missing dates: 0


## 4. Integrate every product without a 59-million-row melt
The complete calculation pivots weekly price once and processes products in blocks. This preserves the exact join logic while controlling memory.

In [5]:
calendar_days = calendar.set_index('d').loc[day_cols].reset_index()
units = sales[day_cols].to_numpy(dtype=np.float32)
row_index = pd.MultiIndex.from_frame(sales[['store_id', 'item_id']])
weeks = calendar_days.wm_yr_wk.drop_duplicates().tolist()
price_wide = prices.pivot(index=['store_id', 'item_id'], columns='wm_yr_wk', values='sell_price').reindex(index=row_index, columns=weeks)
week_position = {week: i for i, week in enumerate(weeks)}
day_week_positions = np.array([week_position[w] for w in calendar_days.wm_yr_wk])

daily_revenue = np.zeros(len(day_cols), dtype=np.float64)
row_revenue = np.zeros(len(sales), dtype=np.float64)
missing_price_sales = 0
for start in range(0, len(sales), 1000):
    stop = min(start + 1000, len(sales))
    block_units = units[start:stop]
    block_prices = price_wide.iloc[start:stop].to_numpy(dtype=np.float32)[:, day_week_positions]
    missing_price_sales += int(((block_units > 0) & np.isnan(block_prices)).sum())
    block_revenue = block_units * np.nan_to_num(block_prices, nan=0.0)
    daily_revenue += block_revenue.sum(axis=0)
    row_revenue[start:stop] = block_revenue.sum(axis=1)

print('Positive sales without price:', missing_price_sales)


Positive sales without price: 0


## 5. Create trusted analytical tables

In [6]:
daily = calendar_days[['date', 'year', 'month', 'weekday', 'event_name_1', 'event_type_1']].copy()
daily['units'] = units.sum(axis=0)
daily['estimated_revenue'] = daily_revenue
daily['average_selling_price'] = daily.estimated_revenue.div(daily.units).replace([np.inf], np.nan)
daily['moving_average_28'] = daily.units.rolling(28, min_periods=1).mean()

product = sales[['item_id', 'cat_id', 'dept_id']].copy()
product['units'] = units.sum(axis=1)
product['estimated_revenue'] = row_revenue
product = product.groupby(['item_id', 'cat_id', 'dept_id'], as_index=False).sum().sort_values('estimated_revenue', ascending=False)
product['revenue_share'] = product.estimated_revenue / product.estimated_revenue.sum()
product['cumulative_revenue_share'] = product.revenue_share.cumsum()
product['abc_class'] = np.select([product.cumulative_revenue_share <= .80, product.cumulative_revenue_share <= .95], ['A', 'B'], default='C')

store_summary = pd.DataFrame([
    {'store_id': store, 'units': units[sales.store_id.eq(store)].sum(), 'estimated_revenue': row_revenue[sales.store_id.eq(store)].sum()}
    for store in sorted(sales.store_id.unique())
])
category_summary = pd.DataFrame([
    {'cat_id': cat, 'units': units[sales.cat_id.eq(cat)].sum(), 'estimated_revenue': row_revenue[sales.cat_id.eq(cat)].sum()}
    for cat in sorted(sales.cat_id.unique())
])

display(pd.Series({'Units sold': daily.units.sum(), 'Estimated revenue': daily.estimated_revenue.sum(), 'Products': len(product), 'Stores': len(store_summary)}).to_frame('value'))


,value
Units sold,6.692717e+07
Estimated revenue,1.915775e+08
Products,3.049000e+03
Stores,1.000000e+01


## 6. Executive visual analysis

In [7]:
fig = go.Figure([
    go.Scatter(x=daily.date, y=daily.units, name='Daily units', opacity=.25),
    go.Scatter(x=daily.date, y=daily.moving_average_28, name='28-day average', line={'width': 3}),
])
fig.update_layout(title='Five-year demand and trend')
fig.show()
px.bar(store_summary.sort_values('estimated_revenue'), x='estimated_revenue', y='store_id', orientation='h', title='Estimated revenue by store').show()
px.bar(category_summary, x='cat_id', y='estimated_revenue', color='cat_id', title='Estimated revenue by category').show()
p = product.reset_index(drop=True).copy(); p['product_share'] = (p.index + 1) / len(p)
px.line(p, x='product_share', y='cumulative_revenue_share', title='Product revenue Pareto curve').show()
display(product.head(20))


,item_id,cat_id,dept_id,units,estimated_revenue,revenue_share,cumulative_revenue_share,abc_class
1198,FOODS_3_586,FOODS,FOODS_3,932236.0,1.482295e+06,0.007737,0.007737,A
732,FOODS_3_120,FOODS,FOODS_3,290132.0,1.444850e+06,0.007542,0.015279,A
702,FOODS_3_090,FOODS,FOODS_3,1017916.0,1.377664e+06,0.007191,0.022470,A
814,FOODS_3_202,FOODS,FOODS_3,300529.0,1.272487e+06,0.006642,0.029112,A
1199,FOODS_3_587,FOODS,FOODS_3,402159.0,9.913073e+05,0.005174,0.034287,A
864,FOODS_3_252,FOODS,FOODS_3,573723.0,8.718064e+05,0.004551,0.038838,A
1167,FOODS_3_555,FOODS,FOODS_3,497881.0,7.890094e+05,0.004118,0.042956,A
1056,FOODS_3_444,FOODS,FOODS_3,109137.0,7.191292e+05,0.003754,0.046710,A
1782,HOBBIES_1_354,HOBBIES,HOBBIES_1,31143.0,7.119376e+05,0.003716,0.050426,A
94,FOODS_1_096,FOODS,FOODS_1,96007.0,6.700772e+05,0.003498,0.053924,A


## Limitations
M5 contains aggregated unit demand, not transactions, customers or basket IDs. Revenue is therefore estimated as units multiplied by weekly selling price; true average ticket cannot be calculated.